In [1]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold Dim Property")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")


# ==================================================
# TEST
# ==================================================

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 16:57:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/07 16:57:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 3.4.0
Master: local[2]
Test: 1


In [2]:
print("Test:", spark.range(1).count())


Test: 1


In [3]:
# ==================================================
# LOAD PLUTO SILVER
# ==================================================

PLUTO_PATH = minio_path(
    "silver/pluto/version=26v2"
)

pluto_df = (
    spark.read
    .parquet(PLUTO_PATH)
)

print("PLUTO loaded successfully")
print("PLUTO rows:", pluto_df.count())

26/09/07 16:58:24 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


PLUTO loaded successfully
PLUTO rows: 858284


In [5]:
# ==================================================
# CREATE DIM_PROPERTY
# Grain: 1 row = 1 Property / BBL
# ==================================================

dim_property = (
    pluto_df

    .filter(
        F.col("bbl").isNotNull()
        & (F.trim(F.col("bbl")) != "")
    )

    # Internal canonical Property ID
    .withColumn(
        "property_id",
        F.concat(
            F.lit("PROP:"),
            F.col("bbl")
        )
    )

    .select(
        "property_id",
        "bbl",

        F.col("address").alias("property_address"),
        "borough",
        "zipcode",

        "latitude",
        "longitude",

        "landuse",
        "bldgclass",

        "yearbuilt",
        "yearalter1",
        "yearalter2",

        "numbldgs",
        "numfloors",
        "unitsres",
        "unitstotal",

        "lotarea",
        "bldgarea",
        "resarea",
        "comarea",

        "ownername",
        "ownertype",

        "snapshot_version"
    )

    .dropDuplicates(
        ["property_id"]
    )
)

In [6]:
print(
    "dim_property rows:",
    dim_property.count()
)

print(
    "Distinct property_id:",
    dim_property
    .select("property_id")
    .distinct()
    .count()
)

print(
    "Distinct BBL:",
    dim_property
    .select("bbl")
    .distinct()
    .count()
)

dim_property rows: 858284


Distinct property_id: 858284


Distinct BBL: 858284


In [7]:
# ==================================================
# SAVE DIM_PROPERTY TO GOLD
# ==================================================

DIM_PROPERTY_PATH = minio_path(
    "gold/data_model/dim_property"
)

(
    dim_property
    .write
    .mode("overwrite")
    .parquet(DIM_PROPERTY_PATH)
)

print("dim_property saved successfully")
print("Path:", DIM_PROPERTY_PATH)

26/09/07 16:59:56 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


dim_property saved successfully
Path: s3a://nyc-building-risk/gold/data_model/dim_property


In [8]:
# ==================================================
# LOAD BUILDING IDENTITY GOLD
# ==================================================

BUILDING_IDENTITY_PATH = minio_path(
    "gold/building_identity/final"
)

building_identity_df = (
    spark.read
    .parquet(BUILDING_IDENTITY_PATH)
)

print("Building Identity loaded successfully")
print(
    "Buildings:",
    building_identity_df.count()
)

Building Identity loaded successfully
Buildings: 197958


In [10]:
# ==================================================
# CREATE DIM_BUILDING
# Grain: 1 row = 1 physical Building / BIN
# ==================================================

dim_building = (
    building_identity_df

    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )

    # Internal canonical Building ID
    .withColumn(
        "building_id",
        F.concat(
            F.lit("BLD:"),
            F.col("bin")
        )
    )

    # Link Building to Property
    .withColumn(
        "property_id",
        F.when(
            F.col("resolved_bbl").isNotNull(),
            F.concat(
                F.lit("PROP:"),
                F.col("resolved_bbl")
            )
        )
    )

    .select(
        "building_id",
        "bin",

        "property_id",
        "resolved_bbl",
        "current_bbl",

        "current_address",
        "borough",

        "latitude",
        "longitude",

        "bbl_aliases",
        "address_aliases",

        "identity_status",
        "match_method",
        "match_confidence",
        "resolution_status"
    )

    .dropDuplicates(
        ["building_id"]
    )
)

In [11]:
print(
    "dim_building rows:",
    dim_building.count()
)

print(
    "Distinct building_id:",
    dim_building
    .select("building_id")
    .distinct()
    .count()
)

print(
    "Buildings with property_id:",
    dim_building
    .filter(
        F.col("property_id").isNotNull()
    )
    .count()
)

print(
    "Buildings without property_id:",
    dim_building
    .filter(
        F.col("property_id").isNull()
    )
    .count()
)

dim_building rows: 197958
Distinct building_id: 197958
Buildings with property_id: 197459
Buildings without property_id: 499


In [12]:
# ==================================================
# SAVE DIM_BUILDING TO GOLD
# ==================================================

DIM_BUILDING_PATH = minio_path(
    "gold/data_model/dim_building"
)

(
    dim_building
    .write
    .mode("overwrite")
    .parquet(DIM_BUILDING_PATH)
)

print("dim_building saved successfully")
print("Path:", DIM_BUILDING_PATH)

dim_building saved successfully
Path: s3a://nyc-building-risk/gold/data_model/dim_building
